# Contagion threshold sensitivity analysis
Sensitivity analysis testing how the distribution of activation thresholds influences the complex
contagion results in our main simulations.

We re-use the social networks already generated and saved by `run_simulations_single_parameter_sweep.ipynb`,
so no new network formation happens here. On each of those networks we run complex contagion under five
threshold distributions:

- `uniform`: our baseline, thresholds drawn from U(0, 1).
- `beta_2_2`: Beta(2, 2), a symmetric hump centered on 0.5 with little mass at the extremes.
- `beta_2_4`: Beta(2, 4), right-skewed with a mean of 1/3, so most nodes have low thresholds.
- `beta_4_2`: Beta(4, 2), left-skewed with a mean of 2/3, so most nodes have high thresholds.
- `uniform_min_1_over_k`: U(0, 1), but each node's threshold is floored at 1 / k_i (its degree), so every
  node needs at least two active neighbors to adopt. Degree-1 nodes therefore can never adopt.

All distributions are run on the same network with the same set of initially active seeds, so the
comparison is paired. Only complex contagion is simulated here, since simple contagion has no thresholds.

In [10]:
# Add project root to Python path
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root))

# Import necessary libraries
import joblib
import networkx as nx
import numpy as np
import pandas as pd
import psutil

from src.population_density_networks import contagion

## Modeling Parameters

In [12]:
# ----- GENERAL -----
# Save paths
OUTPUT_DIR = '../../data_derived'
NETWORK_DIR = f'{OUTPUT_DIR}/full_social_networks'
SAVE_PATH = f'{OUTPUT_DIR}/sensitivity_analysis/contagion_threshold_sensitivity_results.csv'

# Simulation
N_JOBS = psutil.cpu_count(logical=True) - 1


# ----- SAVED NETWORKS -----
# These must match the parameters used to generate the networks in `full_social_networks/`
N = 1000
K_CAP_MEAN = 50
K_CAP_SD = 0.5 * K_CAP_MEAN
RADIUS = 1.0
DENSITIES = np.logspace(-5, 5, 21, base=10)
NETWORK_REPLICATES = 50


# ----- CONTAGION MODEL -----
# Threshold distributions to compare
# THRESHOLD_DISTRIBUTIONS = ['uniform', 'beta_2_2', 'beta_2_4', 'beta_4_2', 'uniform_min_1_over_k']
THRESHOLD_DISTRIBUTIONS = ['uniform', 'uniform_min_1_over_k']


# Beta shape parameters (alpha, beta) keyed by distribution name
BETA_PARAMETERS = {
    'beta_2_2': (2, 2),
    'beta_2_4': (2, 4),
    'beta_4_2': (4, 2),
}

# Complex contagion model parameters
COMPLEX_INITIAL_INFECTED = 0.05

# Simulation parameters
CONTAGION_SIMULATION_ROUNDS = 100
CONTAGION_SIMULATION_REPLICATES = 50  # per simulated network, per threshold distribution

## Threshold Distributions and Seeding

In [13]:
#####################
# Threshold distributions
#####################
def draw_thresholds(
    n: int,
    distribution: str,
    degrees: np.ndarray
) -> np.ndarray:
    """
    Draws a complex contagion activation threshold for each node from a given threshold distribution.

    Args:
    -----
    n :             Number of nodes to draw thresholds for.
    distribution :  Threshold distribution to draw from (an entry of THRESHOLD_DISTRIBUTIONS).
    degrees :       Degree of each node, used by degree-dependent distributions.

    Returns:
    --------
    Activation threshold for each node.
    """
    # Baseline: thresholds spread evenly over the unit interval
    if distribution == 'uniform':
        return np.random.uniform(low=0.0, high=1.0, size=n)

    # Beta thresholds: (2, 2) is symmetric around 0.5, (2, 4) is skewed toward low thresholds,
    # and (4, 2) is skewed toward high ones
    if distribution in BETA_PARAMETERS:
        alpha, beta = BETA_PARAMETERS[distribution]
        return np.random.beta(a=alpha, b=beta, size=n)

    # Uniform thresholds floored at 1 / k_i: since activation requires the active fraction to strictly
    # exceed the threshold, every node needs at least two active neighbors (isolated nodes left as drawn)
    if distribution == 'uniform_min_1_over_k':
        thresholds = np.random.uniform(low=0.0, high=1.0, size=n)
        min_thresholds = np.divide(
            1.0,
            degrees,
            out=np.zeros(n, dtype=float),
            where=degrees > 0
        )
        return np.maximum(thresholds, min_thresholds)

    raise ValueError(f"Unknown threshold distribution: '{distribution}'")


def draw_initial_active(
    n: int,
    initial_active: float
) -> np.ndarray:
    """
    Draws the set of initially active nodes for a contagion simulation.

    Args:
    -----
    n :                 Number of nodes in the network.
    initial_active :    Fraction of the population that starts active.

    Returns:
    --------
    Initial state of each node (0 = susceptible, 1 = active).
    """
    # Bernoulli seeding, matching ContagionModel so this baseline stays comparable to the main simulations
    return np.random.choice([0, 1], size=n, p=[1 - initial_active, initial_active])


def seed_contagion_model(
    contagion_model: contagion.ContagionModel,
    initial_states: np.ndarray
) -> None:
    """
    Replaces the initial states a contagion model drew on construction with a given set of seeds.

    Args:
    -----
    contagion_model :   Initialized contagion model.
    initial_states :    Initial state of each node (0 = susceptible, 1 = active).
    """
    # The model draws its own seeds in __init__, so both the states and the t=0 row of the
    # time series have to be overwritten to hand every threshold distribution identical seeds
    contagion_model.individual_states = initial_states.copy()
    contagion_model.state_time_series = pd.DataFrame({
        'time': [0],
        'susceptible': [np.sum(initial_states == 0)],
        'infected': [np.sum(initial_states == 1)],
        'pct_infected': [np.sum(initial_states == 1) / initial_states.size],
    })
    return None

## Load Saved Social Networks
The networks were saved as edgelists, so we rebuild each adjacency matrix over the full node set
(isolated nodes never appear in an edgelist and would otherwise be dropped).

In [14]:
#####################
# Load a saved network
#####################
def load_social_network(
    network_dir: str,
    density: float,
    replicate: int,
    n: int
) -> pd.DataFrame:
    """
    Loads a previously saved social network edgelist as an adjacency matrix.

    Args:
    -----
    network_dir :   Directory containing the saved edgelists.
    density :       Population density the network was generated at.
    replicate :     Network replicate number.
    n :             Number of nodes in the network.

    Returns:
    --------
    Adjacency matrix of the social network, indexed 0 to n-1.
    """
    edgelist = pd.read_csv(f'{network_dir}/edgelist-density_{density}-replicate_{replicate}.csv')

    # Rebuild over the full node set so isolated nodes are retained as empty rows/columns
    g = nx.from_pandas_edgelist(edgelist, source='source', target='target')
    g.add_nodes_from(range(n))
    return nx.to_pandas_adjacency(g, nodelist=range(n), dtype=int)

## Verify the Threshold Distributions
Quick check that each distribution has the shape we intend before running the full sweep. Since what
actually drives activation is the number of active neighbors, we also translate thresholds into that
scale: a node activates when the active fraction of its neighbors exceeds its threshold, so the smallest
number of active neighbors that can activate node i is `floor(threshold_i * k_i) + 1`.

In [15]:
#####################
# Compare threshold distributions on one example network
#####################
# Pull a mid-density network and its degree sequence
example_network = load_social_network(
    network_dir=NETWORK_DIR,
    density=DENSITIES[10],
    replicate=0,
    n=N
)
example_degrees = example_network.sum(axis=1).values

# Draw thresholds under each distribution and convert them to the number of active neighbors needed
threshold_check = pd.concat(
    [
        pd.DataFrame({
            'threshold_distribution': distribution,
            'degree': example_degrees,
            'threshold': draw_thresholds(n=N, distribution=distribution, degrees=example_degrees),
        })
        for distribution in THRESHOLD_DISTRIBUTIONS
    ],
    ignore_index=True
)
threshold_check['neighbors_needed'] = np.floor(threshold_check['threshold'] * threshold_check['degree']) + 1

# Summarize over connected nodes (isolated nodes can never activate under any distribution)
print(f"Example network: density = {DENSITIES[10]:.3g}, mean degree = {example_degrees.mean():.1f}, "
      f"nodes with degree < 2 = {(example_degrees < 2).sum()}")

(
    threshold_check
    .loc[threshold_check['degree'] > 0]
    .groupby('threshold_distribution', as_index=False)
    .agg(
        min_threshold=('threshold', 'min'),
        mean_threshold=('threshold', 'mean'),
        sd_threshold=('threshold', 'std'),
        max_threshold=('threshold', 'max'),
        median_neighbors_needed=('neighbors_needed', 'median')
    )
)

Example network: density = 1, mean degree = 47.2, nodes with degree < 2 = 0


,threshold_distribution,min_threshold,mean_threshold,sd_threshold,max_threshold,median_neighbors_needed
0,uniform,0.000763,0.498814,0.293452,0.999328,21.0
1,uniform_min_1_over_k,0.016129,0.500235,0.290265,0.999802,21.0


## Run Contagion Simulations Across Threshold Distributions

In [16]:
#####################
# Function to run complex contagion on a saved network under each threshold distribution
#####################
def run_threshold_sensitivity_simulation(
    # Network parameters
    density: float,
    replicate: int,
    n: int,
    k_cap_mean: float,
    k_cap_sd: float,
    network_dir: str,
    # Contagion model parameters
    threshold_distributions: list[str],
    complex_initial_infected: float,
    contagion_simulation_rounds: int,
    contagion_simulation_replicates: int
) -> pd.DataFrame:
    """
    Runs complex contagion on one saved network under each threshold distribution.

    Args:
    -----
    density :                           Population density the network was generated at.
    replicate :                         Network replicate number.
    n :                                 Number of nodes in the network.
    k_cap_mean :                        Average maximum degree limit used to generate the network.
    k_cap_sd :                          Standard deviation of the maximum degree limit used to generate the network.
    network_dir :                       Directory containing the saved edgelists.
    threshold_distributions :           Threshold distributions to compare.
    complex_initial_infected :          Fraction of the population that starts active.
    contagion_simulation_rounds :       Number of time steps per contagion simulation.
    contagion_simulation_replicates :   Number of contagion replicates per threshold distribution.

    Returns:
    --------
    Contagion spread statistics for every replicate and threshold distribution.
    """
    # Load the pre-generated network
    network = load_social_network(
        network_dir=network_dir,
        density=density,
        replicate=replicate,
        n=n
    )
    degrees = network.sum(axis=1).values

    # Run paired replicates: within a replicate every threshold distribution sees the same seeds
    contagion_results = []
    for contagion_replicate in range(contagion_simulation_replicates):
        initial_states = draw_initial_active(n=n, initial_active=complex_initial_infected)

        for distribution in threshold_distributions:

            # Set up the complex contagion model with thresholds from this distribution
            thresholds = draw_thresholds(n=n, distribution=distribution, degrees=degrees)
            complex_contagion_model = contagion.ComplexContagionModel(
                network=network,
                initial_infected=complex_initial_infected,
                thresholds=thresholds
            )
            seed_contagion_model(complex_contagion_model, initial_states)

            # Simulate and analyze spread
            complex_contagion_model.run_simulation(time_steps=contagion_simulation_rounds)
            result = {
                'threshold_distribution': distribution,
                'contagion_replicate': contagion_replicate,
            }
            result.update(contagion.analyze_contagion_results(complex_contagion_model))
            contagion_results.append(result)

    # Label with the network-level parameters
    contagion_results = pd.DataFrame(contagion_results)
    contagion_results.insert(1, 'population_density', density)
    contagion_results.insert(2, 'k_cap_mean', k_cap_mean)
    contagion_results.insert(3, 'k_cap_sd', k_cap_sd)
    contagion_results.insert(4, 'network_replicate', replicate)

    return contagion_results

In [17]:
#####################
# Run simulations
#####################
# Generate all parameter combinations
param_combinations = [
    (
        density,
        replicate,
        N,
        K_CAP_MEAN,
        K_CAP_SD,
        NETWORK_DIR,
        THRESHOLD_DISTRIBUTIONS,
        COMPLEX_INITIAL_INFECTED,
        CONTAGION_SIMULATION_ROUNDS,
        CONTAGION_SIMULATION_REPLICATES,
    )
    for density in DENSITIES
    for replicate in range(NETWORK_REPLICATES)
]

n_runs = len(param_combinations) * len(THRESHOLD_DISTRIBUTIONS) * CONTAGION_SIMULATION_REPLICATES
print(f"🚀 Running {len(param_combinations)} networks x {len(THRESHOLD_DISTRIBUTIONS)} threshold "
      f"distributions x {CONTAGION_SIMULATION_REPLICATES} replicates = {n_runs} contagion simulations...")

# Run simulations in parallel
parallel_jobs = min(N_JOBS, len(param_combinations))
results_list = joblib.Parallel(n_jobs=parallel_jobs, verbose=10)(
    joblib.delayed(run_threshold_sensitivity_simulation)(*params) for params in param_combinations
)

# Convert results to DataFrame
results = pd.concat(results_list, ignore_index=True)

🚀 Running 1050 networks x 2 threshold distributions x 50 replicates = 105000 contagion simulations...


[Parallel(n_jobs=13)]: Using backend LokyBackend with 13 concurrent workers.
[Parallel(n_jobs=13)]: Done   6 tasks      | elapsed:   12.4s
[Parallel(n_jobs=13)]: Done  15 tasks      | elapsed:   25.1s
[Parallel(n_jobs=13)]: Done  24 tasks      | elapsed:   26.1s
[Parallel(n_jobs=13)]: Done  35 tasks      | elapsed:   37.8s
[Parallel(n_jobs=13)]: Done  46 tasks      | elapsed:   49.4s
[Parallel(n_jobs=13)]: Done  59 tasks      | elapsed:  1.0min
[Parallel(n_jobs=13)]: Done  72 tasks      | elapsed:  1.2min
[Parallel(n_jobs=13)]: Done  87 tasks      | elapsed:  1.5min
[Parallel(n_jobs=13)]: Done 102 tasks      | elapsed:  1.7min
[Parallel(n_jobs=13)]: Done 119 tasks      | elapsed:  2.1min
[Parallel(n_jobs=13)]: Done 136 tasks      | elapsed:  2.3min
[Parallel(n_jobs=13)]: Done 155 tasks      | elapsed:  2.5min
[Parallel(n_jobs=13)]: Done 174 tasks      | elapsed:  2.9min
[Parallel(n_jobs=13)]: Done 195 tasks      | elapsed:  3.1min
[Parallel(n_jobs=13)]: Done 216 tasks      | elapsed:  

In [18]:
results

,threshold_distribution,population_density,k_cap_mean,k_cap_sd,network_replicate,contagion_replicate,final_infected_fraction,time_to_majority,reached_majority_spread,max_slope
0,uniform,0.00001,50,25.0,0,0,0.589,14.0,True,0.058
1,uniform_min_1_over_k,0.00001,50,25.0,0,0,0.672,7.0,True,0.086
2,uniform,0.00001,50,25.0,0,1,0.689,13.0,True,0.054
3,uniform_min_1_over_k,0.00001,50,25.0,0,1,0.638,13.0,True,0.048
4,uniform,0.00001,50,25.0,0,2,0.714,12.0,True,0.049
...,...,...,...,...,...,...,...,...,...,...
104995,uniform_min_1_over_k,100000.00000,50,25.0,49,47,0.738,12.0,True,0.071
104996,uniform,100000.00000,50,25.0,49,48,0.421,NaN,False,0.043
104997,uniform_min_1_over_k,100000.00000,50,25.0,49,48,0.840,11.0,True,0.055
104998,uniform,100000.00000,50,25.0,49,49,0.804,16.0,True,0.046


## Compare Threshold Distributions

In [19]:
#####################
# Summarize spread by density and threshold distribution
#####################
threshold_comparison = (
    results
    .groupby(['threshold_distribution', 'population_density'], as_index=False)
    .agg(
        mean_final_infected_fraction=('final_infected_fraction', 'mean'),
        sd_final_infected_fraction=('final_infected_fraction', 'std'),
        pct_reached_majority=('reached_majority_spread', 'mean'),
        mean_time_to_majority=('time_to_majority', 'mean'),
        mean_max_slope=('max_slope', 'mean')
    )
)
threshold_comparison.head(50)

,threshold_distribution,population_density,mean_final_infected_fraction,sd_final_infected_fraction,pct_reached_majority,mean_time_to_majority,mean_max_slope
0,uniform,0.000010,0.604252,0.173014,0.7120,15.814607,0.050462
1,uniform,0.000032,0.604191,0.178315,0.7028,15.790552,0.050497
2,uniform,0.000100,0.604483,0.176676,0.7104,15.686374,0.050526
3,uniform,0.000316,0.600227,0.175980,0.6968,15.861653,0.050195
4,uniform,0.001000,0.600352,0.175039,0.7028,15.735913,0.050417
5,uniform,0.003162,0.605090,0.181246,0.7052,15.829268,0.050349
6,uniform,0.010000,0.604290,0.181277,0.7076,15.737705,0.050850
7,uniform,0.031623,0.605494,0.180943,0.7052,15.678389,0.050360
8,uniform,0.100000,0.612085,0.179004,0.7176,15.699554,0.050662
9,uniform,0.316228,0.618261,0.180905,0.7300,15.729315,0.050581


## Save results

In [20]:
#####################
# Save data
#####################
results.to_csv(SAVE_PATH, index=False)

In [21]:
results = pd.read_csv(SAVE_PATH)

#####################
# Summarize spread by density and threshold distribution
#####################
threshold_comparison = (
    results
    .groupby(['threshold_distribution', 'population_density'], as_index=False)
    .agg(
        mean_final_infected_fraction=('final_infected_fraction', 'mean'),
        sd_final_infected_fraction=('final_infected_fraction', 'std'),
        pct_reached_majority=('reached_majority_spread', 'mean'),
        mean_time_to_majority=('time_to_majority', 'mean'),
        mean_max_slope=('max_slope', 'mean')
    )
)
threshold_comparison.tail(50)

threshold_comparison.to_csv('~/Downloads/temp.csv', index=False)